## Paw + wheel wavelet subsample

Subsamples **left-paw wavelets + wheel wavelets** (0.5-8.0 Hz) into one super-session for clustering.
Reads paw wavelets from `paw_wavelets/` and wheel wavelets from `wheel_wavelets/`, merged on `Bin`.

In [1]:
"""
IMPORTS
"""
import os
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.manifold import TSNE

from segmentation_functions import idxs_from_files

In [ ]:
"""
PATHS AND FEATURES (proficient: left paw + wheel wavelets)
"""
base_path = '/home/ines/repositories/representation_learning_variability/paper-individuality/data/'
data_path = base_path + 'design_matrices/'
data_path = base_path + 'design_matrices/1_camera_setup/NM/' 
paw_wavelet_path = base_path + 'paw_wavelets/'      # left paw wavelets live here
wheel_wavelet_path = base_path + 'wheel_wavelets/'  # wheel wavelets live here
paw_wavelet_path = base_path + 'paw_wavelets/'      # left paw wavelets live here
wheel_wavelet_path = base_path + 'wheel_wavelets/'  # wheel wavelets live here

all_files = os.listdir(data_path)
design_matrices = [item for item in all_files if 'design_matrix' in item and 'standardized' not in item]
idxs, mouse_names = idxs_from_files(design_matrices)

# left paw + wheel wavelet features (0.5-8.0 Hz)
l_paw_cols = ['l_paw_x0.5', 'l_paw_x1.0', 'l_paw_x2.0', 'l_paw_x4.0', 'l_paw_x8.0',
              'l_paw_y0.5', 'l_paw_y1.0', 'l_paw_y2.0', 'l_paw_y4.0', 'l_paw_y8.0']
wheel_cols = ['avg_wheel_vel0.5', 'avg_wheel_vel1.0', 'avg_wheel_vel2.0', 'avg_wheel_vel4.0', 'avg_wheel_vel8.0']
var_interest = l_paw_cols + wheel_cols

In [12]:
# Sessions that have BOTH a paw and a wheel wavelet file
paw_files = os.listdir(paw_wavelet_path)
wheel_files = os.listdir(wheel_wavelet_path)
sessions_to_process = []
for m, mat in enumerate(idxs):
    mouse_name = mat[37:]
    session = mat[:36]
    if ('paw_vel_wavelets_' + session + '_' + mouse_name in paw_files) and \
       ('wheel_vel_wavelets_' + session + '_' + mouse_name in wheel_files):
        sessions_to_process.append((mouse_name, session))
print(len(sessions_to_process))


0


In [5]:
# Subsample each session (t-SNE + KDE-weighted, as in 3.2) and concatenate into a supersession
concatenated_subsampled = np.array([])
for m, mat in enumerate(sessions_to_process):

    mouse_name = mat[0]
    session = mat[1]
    # Load paw + wheel wavelets and merge on Bin (same samples)
    paw = pd.read_parquet(paw_wavelet_path + 'paw_vel_wavelets_' + str(session) + '_' + mouse_name)
    wheel = pd.read_parquet(wheel_wavelet_path + 'wheel_vel_wavelets_' + str(session) + '_' + mouse_name)
    merged = paw[['Bin'] + l_paw_cols].merge(wheel[['Bin'] + wheel_cols], on='Bin', how='inner')

    data = merged[var_interest].dropna().to_numpy()

    # Randomly subsample
    n_samples = 20000
    if data.shape[0] < n_samples:
        continue
    sampled_data = data[np.random.choice(data.shape[0], n_samples, replace=False)]

    # t-SNE + KDE-weighted resample
    X = stats.zscore(sampled_data, axis=0)
    X_embedded = TSNE(n_components=2, learning_rate='auto', init='random', perplexity=32).fit_transform(X)
    values = X_embedded.T.copy()
    kernel = stats.gaussian_kde(values)
    sample_prob = kernel(values)
    sample_prob = sample_prob / np.sum(sample_prob)
    resampled_data = sampled_data[np.random.choice(sampled_data.shape[0], size=2000, p=sample_prob, replace=False)]

    # z-score within session and concatenate
    zscored_data = stats.zscore(resampled_data, axis=0, nan_policy='omit')
    if len(concatenated_subsampled) == 0:
        concatenated_subsampled = zscored_data.copy()
    else:
        concatenated_subsampled = np.vstack([concatenated_subsampled, zscored_data])

print(concatenated_subsampled.shape)


KeyboardInterrupt: 

In [15]:
# Save the joint supersession (columns = var_interest = l_paw_cols + wheel_cols)
super_name = 'session_zscored_supersession_wavelets_left_paw_wheel'
np.save(open(base_path + super_name, 'wb'), concatenated_subsampled)
print('saved', base_path + super_name, concatenated_subsampled.shape)


saved /home/ines/repositories/representation_learning_variability/paper-individuality/data/session_zscored_supersession_wavelets_left_paw_wheel (638000, 15)
